# 공공문서 RAG 검색 평가 실습

검색 평가에 집중할 수 있도록 PDF 로드, 메타데이터 정규화, 문서 분할 코드를 제공한다.

### 실습 목표

- 제공된 전처리 결과를 사용해 벡터 저장소를 구성한다.
- File Hit@k, Page Hit@k, MRR을 계산한다.
- Similarity Search와 MMR의 검색 성능을 비교한다.
- 실패 문항의 검색 결과를 확인하고 원인을 분석한다.

### 제공 데이터

- 검색 대상 문서: `data/public/*.pdf`
- 평가 데이터: `data/public/public_paragraph_golden_set.json`

golden set의 `question`은 검색 질의, `target_answer`는 기준 답변, `target_file_name`과 `target_page_no`는 정답 위치를 나타낸다.

`PyPDFLoader`가 만든 `source`에는 `data/public/파일명.pdf`처럼 경로가 포함될 수 있지만, golden set의 `target_file_name`에는 파일명만 들어 있다. 두 값을 바로 비교할 수 있도록 `Path(source).name`을 사용해 경로를 제거한다. 이 코드는 아래에 제공되어 있으므로 별도로 구현하지 않는다.

또한 `PyPDFLoader`의 `page`는 첫 페이지를 0으로 나타내지만 golden set의 `target_page_no`는 첫 페이지를 1로 나타낸다. 비교에 사용할 `page_no`에는 `page + 1`을 저장한다.

In [1]:
import json
import time
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

DATA_DIR = Path("data/public")
GOLDEN_SET_PATH = DATA_DIR / "public_paragraph_golden_set.json"
EMBEDDING_MODEL_NAME = "gemini-embedding-2"

C:\Users\김나린\AppData\Local\Temp\ipykernel_29396\1450973640.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Golden set 확인

평가 문항 수와 필드 구성을 확인한다.

In [2]:
eval_cases = json.loads(GOLDEN_SET_PATH.read_text(encoding="utf-8"))

print(f"평가 문항: {len(eval_cases)}개")
print(json.dumps(eval_cases[:5], ensure_ascii=False, indent=2))

평가 문항: 35개
[
  {
    "domain": "public",
    "question": "2023년 국가재정운용계획 수립 절차 중, 분야별·지역별 예산협의회 및 간담회 개최 시기와 국회 제출 시기는 언제인가요?",
    "target_answer": "2023년 국가재정운용계획 수립 절차에 따르면, 분야별·지역별 예산협의회 및 간담회는 2023년 4월부터 8월까지 개최되고, 국회 제출은 2023년 9월 1일에 이루어집니다.",
    "target_file_name": "2023_2027 국가재정운용계획 주요내용.pdf",
    "target_page_no": 3,
    "context_type": "paragraph"
  },
  {
    "domain": "public",
    "question": "불요불급한 자산을 처분하여 회수하는 자금은 어떻게 활용되는지, 그리고 공공기관의 재무건전성 관리 강화를 위해 '22~'26년 재정건전화계획 수정계획을 수립하는 것 외에 어떤 방법이 적용되는지 설명해주세요.",
    "target_answer": "불요불급한 자산을 처분하여 회수된 자금은 재무건전성 제고 등에 활용됩니다. 또한, 재무건전성 관리 강화를 위해 대내외 여건 변화를 반영하여 재무위험기관의 '22~'26년 재정건전화계획의 수정계획을 수립하고, 공공기관의 혁신계획과 재정건전화계획 등을 반영한 중장기 재무관리계획을 수립하여 국회에 제출합니다.",
    "target_file_name": "2023_2027 국가재정운용계획 주요내용.pdf",
    "target_page_no": 22,
    "context_type": "paragraph"
  },
  {
    "domain": "public",
    "question": "북한의 위협에 대응하기 위해 강화해야하는 한국형 3축체계와는 무엇이며, 이를 통해 향상시키려는 국방 분야의 세부적인 전투능력은 무엇인가요?",
    "target_answer": "한국형 3축체계는 북

### PDF 로드

모든 PDF를 페이지 단위로 불러온다. `PyPDFLoader.load()`의 결과에서 문서 하나는 PDF의 한 페이지에 해당한다.

불러온 페이지의 메타데이터는 다음과 같이 정리한다.

```python
{
    "source": "2024년 행정안전부 업무계획.pdf",
    "page": 0,
    "page_no": 1,
}
```

`source`는 golden set과 비교하기 위한 파일명이고, `page_no`는 golden set과 비교하기 위한 페이지 번호이다. 원래 로더가 제공한 `page`도 그대로 유지한다.

In [3]:
page_documents = []

for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    pages = PyPDFLoader(pdf_path).load()

    for page in pages:
        # source에는 전체 경로 대신 golden set과 같은 파일명만 저장한다.
        page.metadata["source"] = Path(page.metadata["source"]).name
        # PyPDFLoader의 page는 0부터 시작하므로 사람이 보는 페이지 번호로 바꾼다.
        page.metadata["page_no"] = page.metadata["page"] + 1

    page_documents.extend(pages)

print(f"로드한 PDF: {len(list(DATA_DIR.glob('*.pdf')))}개")
print(f"로드한 페이지: {len(page_documents)}개")
print(page_documents[0].metadata)

로드한 PDF: 11개
로드한 페이지: 236개
{'producer': 'Hancom PDF 1.3.0.546', 'creator': 'Hwp 2020 11.0.0.8362', 'creationdate': '2024-04-11T11:32:01+09:00', 'moddate': '2024-04-11T11:32:01+09:00', 'pdfversion': '1.4', 'source': '(240411보도자료) 재정동향 4월호.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'page_no': 1}


### 문서 분할

페이지 정보를 유지한 채 문서를 청크로 분할한다. 이 실습에서는 제공된 전처리 결과를 사용한다.

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
)

documents = splitter.split_documents(page_documents)

print(f"생성한 청크: {len(documents)}개")

생성한 청크: 402개


### 벡터 저장소와 검색기 구성

동일한 문서로 Similarity Search와 MMR을 구성한다. 이 실습에서는 별도 저장 경로 없이 `Chroma.from_documents()`로 벡터 저장소를 만들고, 검색기 딕셔너리의 키는 `Similarity`와 `MMR`을 사용한다.

In [6]:
import chromadb

COLLECTION_NAME = "golden_set_brief"
PERSIST_DIR = "./chroma_db"

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL_NAME)

# TODO: vectorstore와 retrievers를 구성하세요.
vectorstore = None
retrievers = {}

client = chromadb.PersistentClient(path=PERSIST_DIR)
existing_names = [c.name for c in client.list_collections()]
if COLLECTION_NAME in existing_names:
    client.delete_collection(COLLECTION_NAME)

# 벡터 스토어 생성 + 문서 저장 — client를 공유하여 일관된 경로 사용
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    client=client,
)

print(f"저장 완료! (컬렉션: {COLLECTION_NAME})")

저장 완료! (컬렉션: golden_set_brief)


In [26]:
from kiwipiepy import Kiwi
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever


FETCH_K = 5

similarity_retriever = vectorstore.as_retriever(search_kwargs={"k": FETCH_K})
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": FETCH_K, "fetch_k": 20},
)

kiwi = Kiwi()


def kiwi_tokenize(text):
    tokens = kiwi.tokenize(text)
    return [
        token.form.lower()
        for token in tokens
        if token.tag.startswith(("N", "V", "M", "X"))
        or token.tag in {"SL", "SN"}
    ]


bm25_retriever = BM25Retriever.from_documents(
    documents,
    preprocess_func=kiwi_tokenize,
    k=FETCH_K,
)
hybrid_retriever = EnsembleRetriever(
    retrievers=[similarity_retriever, bm25_retriever],
    weights=[0.5, 0.5],
)

retrievers = {
    "Similarity": similarity_retriever,
    "MMR": mmr_retriever,
    "BM25": bm25_retriever,
    "Hybrid": hybrid_retriever,
}

### 평가 지표 구현

File Hit은 정답 파일의 검색 여부를, Page Hit은 정답 파일과 페이지가 함께 일치하는지를 확인한다. MRR은 최초로 등장한 정답 페이지 순위의 역수이다.

검색 결과 문서 `doc`과 golden set 문항 `case`는 다음 값으로 비교한다.

```python
파일 일치: doc.metadata["source"] == case["target_file_name"]
페이지 일치: doc.metadata["page_no"] == case["target_page_no"]
정답 페이지: 파일 일치 and 페이지 일치
```

예를 들어 정답 페이지가 검색 결과의 세 번째에 처음 등장하면 RR은 `1 / 3`이다. 상위 k개에 정답이 없으면 Hit은 0이고, 전체 검색 결과에 정답 페이지가 없으면 RR은 0.0이다.

In [ ]:
def file_hit_at_k(docs, case, k):
    # TODO

    return int(any(
        doc.metadata["source"] 
        == case["target_file_name"]
        for doc in docs[:k]
    ))


def page_hit_at_k(docs, case, k):
    # TODO
    return int(any(
        doc.metadata["source"]
        == case["target_file_name"]
        and
        doc.metadata["page_no"] 
        == case["target_page_no"]
        for doc in docs[:k]
    ))


def reciprocal_rank(docs, case):
    # TODO
    for rank, doc in enumerate(
        docs,
        start = 1
    ):
        if(
            doc.metadata["source"] 
            == case["target_file_name"]
            and
            doc.metadata["page_no"] 
            == case["target_page_no"]
        ):
            return 1 / rank

    return 0.0

### 전체 평가

모든 질문을 검색하고 전략별 지표와 실행 시간을 기록한다. latency는 `retriever.invoke()` 한 번에 걸린 시간을 밀리초 단위로 측정한다. 실패 원인을 확인할 수 있도록 검색 결과의 순위, 파일명, 페이지 번호도 함께 저장한다.

In [ ]:
def show_table(items, keys, widths):
    def format_value(value, width):
        if isinstance(value, float):
            text = f"{value:.3f}"
        elif isinstance(value, (list, dict)):
            text = json.dumps(value, ensure_ascii=False)
        else:
            text = str(value)
        text = text.replace("\n", " ")
        return text if len(text) <= width else text[:width - 1] + "…"

    print(" | ".join(f"{key:<{width}}" for key, width in zip(keys, widths)))
    print("-+-".join("-" * width for width in widths))
    for item in items:
        values = [format_value(item.get(key, ""), width) for key, width in zip(keys, widths)]
        print(" | ".join(f"{value:<{width}}" for value, width in zip(values, widths)))


print(f"평가 문항: {len(eval_cases)}개")
show_table(
    eval_cases[:5],
    ["question", "target_answer", "target_file_name", "target_page_no"],
    [36, 42, 30, 14],
)

평가 문항: 35개
question                             | target_answer                              | target_file_name               | target_page_no
-------------------------------------+--------------------------------------------+--------------------------------+---------------
2023년 국가재정운용계획 수립 절차 중, 분야별·지역별 예산협… | 2023년 국가재정운용계획 수립 절차에 따르면, 분야별·지역별 예산협의회 … | 2023_2027 국가재정운용계획 주요내용.pdf    | 3             
불요불급한 자산을 처분하여 회수하는 자금은 어떻게 활용되는지, … | 불요불급한 자산을 처분하여 회수된 자금은 재무건전성 제고 등에 활용됩니다.… | 2023_2027 국가재정운용계획 주요내용.pdf    | 22            
북한의 위협에 대응하기 위해 강화해야하는 한국형 3축체계와는 무… | 한국형 3축체계는 북한의 핵 및 미사일과 같은 다양하고 고도화된 위협에 대… | 2023_2027 국가재정운용계획 주요내용.pdf    | 33            
사업비 정산에 대한 절차 중 국고지원금 집행 시점과 사업종료 후… | 국고지원금은 ’25년 2월 말까지 집행이 완료되어야 하며, 사업기간 종료 … | 2024년+국립대학+육성사업+기본계획.pdf       | 19            
‘24년 대학혁신지원사업에 대한 성과평가용 자료 제출 및 성과평… | 성과평가용 자료 제출 및 성과평가는 '24년 6~7월에 예정되어 있으며, … | 2024년+국립대학+육성사업+기본계획.pdf       | 21            


retrievers = {
    "Similarity": similarity_retriever,
    "MMR": mmr_retriever,
    "BM25": bm25_retriever,
    "Hybrid": hybrid_retriever,
}

In [50]:
def evaluate_retriever(name,retriever,cases,):
    rows = []

    for case in cases:
        started = time.perf_counter()

        docs = retriever.invoke(
            case["question"]
        )

        latency_ms = (
            time.perf_counter() - started
        ) * 1000

        rows.append({
            "strategy": name,
            "question": case["question"],
            "file_hit_1": file_hit_at_k(
                docs, case, 1
            ),
            "page_hit_1": page_hit_at_k(
                docs, case, 1
            ),
            "page_hit_3": page_hit_at_k(
                docs, case, 3
            ),
            "page_hit_5": page_hit_at_k(
                docs, case, 5
            ),
            "reciprocal_rank": reciprocal_rank(
                docs, case
            ),
            "latency_ms": latency_ms,
        })

    return rows

evaluation_results = []
for name, retriever in retrievers.items():
    evaluation_results.extend(
        evaluate_retriever(name, retriever, eval_cases)
    )

show_table(
    evaluation_results,
    ["strategy", "question", "file_hit_1", "page_hit_1", "page_hit_3", "page_hit_5", "reciprocal_rank", "latency_ms"],
    [10, 36, 10, 10, 10, 10, 15, 12],
)

strategy   | question                             | file_hit_1 | page_hit_1 | page_hit_3 | page_hit_5 | reciprocal_rank | latency_ms  
-----------+--------------------------------------+------------+------------+------------+------------+-----------------+-------------
Similarity | 2023년 국가재정운용계획 수립 절차 중, 분야별·지역별 예산협… | 1          | 1          | 1          | 1          | 1.000           | 633.382     
Similarity | 불요불급한 자산을 처분하여 회수하는 자금은 어떻게 활용되는지, … | 1          | 1          | 1          | 1          | 1.000           | 566.681     
Similarity | 북한의 위협에 대응하기 위해 강화해야하는 한국형 3축체계와는 무… | 1          | 1          | 1          | 1          | 1.000           | 534.086     
Similarity | 사업비 정산에 대한 절차 중 국고지원금 집행 시점과 사업종료 후… | 1          | 1          | 1          | 1          | 1.000           | 596.852     
Similarity | ‘24년 대학혁신지원사업에 대한 성과평가용 자료 제출 및 성과평… | 1          | 1          | 1          | 1          | 1.000           | 537.309     
Similarity | 2023년 국가안전시스템 개편에 대한 주요 성과와 함께, 이러한… | 1  

### 결과 비교

검색 전략별 평균 점수를 비교한다. 전략마다 문항 수 `count`와 `file_hit_1`, `page_hit_1`, `page_hit_3`, `page_hit_5`, `reciprocal_rank`, `latency_ms`의 평균을 계산한다. 여러 문항의 `reciprocal_rank` 평균이 MRR이다.

In [51]:
# TODO: 검색 전략별 결과를 요약하세요.
metric_names = [
    "file_hit_1",
    "page_hit_1",
    "page_hit_3",
    "page_hit_5",
    "reciprocal_rank",
    "latency_ms",
]


summary = []

for strategy in retrievers:
    strategy_rows = [
        row for row in evaluation_results
        if row["strategy"] == strategy
    ]
    summary.append({
        "strategy": strategy,
        "count": len(strategy_rows),
        **{
            metric: sum(row[metric] for row in strategy_rows) / len(strategy_rows)
            for metric in metric_names
        },
    })

show_table(
    summary,
    ["strategy", "count", *metric_names],
    [10, 7, 10, 10, 10, 10, 17, 12],
)

strategy   | count   | file_hit_1 | page_hit_1 | page_hit_3 | page_hit_5 | reciprocal_rank   | latency_ms  
-----------+---------+------------+------------+------------+------------+-------------------+-------------
Similarity | 35      | 0.971      | 0.886      | 0.914      | 0.943      | 0.907             | 534.389     
MMR        | 35      | 0.971      | 0.886      | 0.943      | 0.943      | 0.907             | 506.248     
BM25       | 35      | 0.971      | 0.914      | 0.943      | 0.943      | 0.907             | 3.114       
Hybrid     | 35      | 0.971      | 0.914      | 0.943      | 0.943      | 0.910             | 508.166     


### 실패 문항 분석

전체 평가 결과에서 `page_hit_5 == 0`인 행을 확인한다. 각 실패 행에 저장한 `retrieved`를 정답 파일·페이지와 비교하여 같은 파일의 다른 페이지가 검색되었는지, 다른 파일만 검색되었는지 분석한다.

In [52]:
failures = [
    row for row in evaluation_results
    if row["page_hit_5"] == 0
]
show_table(
    failures,
    ["strategy", "question", "target_page_no", "page_hit_1", "page_hit_3", "page_hit_5", "retrieved"],
    [10, 36, 14, 10, 10, 10, 48],
)

strategy   | question                             | target_page_no | page_hit_1 | page_hit_3 | page_hit_5 | retrieved                                       
-----------+--------------------------------------+----------------+------------+------------+------------+-------------------------------------------------
Similarity | 청년농 우대보증 한도를 5억으로 정하고 후계농 정책자금으로 3억… |                | 0          | 0          | 0          |                                                 
Similarity | 금융기관에서 담보 설정시 대출가능 금액이 토지와 시설물에서 각각… |                | 0          | 0          | 0          |                                                 
MMR        | 청년농 우대보증 한도를 5억으로 정하고 후계농 정책자금으로 3억… |                | 0          | 0          | 0          |                                                 
MMR        | 금융기관에서 담보 설정시 대출가능 금액이 토지와 시설물에서 각각… |                | 0          | 0          | 0          |                                                 
BM25       | 청년농 우대보증 한도를 5억으로 정하고 후계농 정책자금으로 3억… |       

### 결과 정리

다음 내용을 작성한다.

- 가장 성능이 좋은 검색 전략과 판단 근거
- File Hit과 Page Hit 결과가 다른 이유
- 실패 문항에서 확인한 검색 결과의 특징
- 성능을 개선하기 위해 변경해 볼 설정

- 모든 전략의 정확도가 거의 비슷하므로, 평균 검색 시간이 가장 짧은 BM25가 가장 효율적으로 보인다

- File Hit은 파일만 확인하지만, Page Hit은 파일과 페이지가 모두 맞아야 하므로 결과가 다를 수 있다.

- 실패 문항은 정답 파일은 찾았지만, 정답 페이지가 상위 5개 안에 들어오지 못했다.

- 개선을 위해 청크 크기와 겹치는 범위, k를 조정하고 검색 전략마다 결과를 다시 비교할 수 있다.